# 12.16 - RAG Evaluation

**Phase:** 12 - LangChain

**Status:** VERIFIED

---

## 1. What Are We Solving?

How do you know if your RAG system is working? Evaluation measures retrieval quality and answer accuracy.

## 2. Why Does This Matter?

Without evaluation, you cannot improve. RAG evaluation catches hallucinations and retrieval failures.

## 3. Prerequisites

- 12.07: RAG with LangChain
- 12.12: Embeddings & vector stores

## 4. Learning Objectives

- Evaluate retrieval quality
- Measure answer faithfulness
- Build evaluation pipelines
- Use LangChain evaluation tools

## 5. Mental Model

RAG evaluation = two parts:
1. Retrieval: did we find the right documents?
2. Generation: is the answer faithful to the documents?

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
print("Libraries loaded.")

Libraries loaded.


## 6. Setup RAG System

In [2]:
llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)

# Sample knowledge base
docs = [
    Document(page_content="Python is a high-level programming language created by Guido van Rossum in 1991.", metadata={"source": "python.txt"}),
    Document(page_content="Machine learning is a subset of AI that learns patterns from data.", metadata={"source": "ml.txt"}),
    Document(page_content="Deep learning uses neural networks with many layers.", metadata={"source": "dl.txt"}),
    Document(page_content="LangChain is a framework for building LLM-powered applications.", metadata={"source": "lc.txt"}),
]

# Simple retrieval
def retrieve(query, docs, k=2):
    scores = []
    query_words = set(query.lower().split())
    for doc in docs:
        doc_words = set(doc.page_content.lower().split())
        overlap = len(query_words & doc_words)
        scores.append((doc, overlap))
    scores.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in scores[:k]]

# RAG chain
rag_prompt = ChatPromptTemplate.from_template(
    "Context:\n{context}\n\nQuestion: {question}\n\nAnswer based on context:"
)

rag_chain = rag_prompt | llm | StrOutputParser()

def rag(query):
    retrieved = retrieve(query, docs)
    context = "\n".join(d.page_content for d in retrieved)
    answer = rag_chain.invoke({"context": context, "question": query})
    return {"answer": answer, "sources": [d.metadata["source"] for d in retrieved]}

result = rag("What is Python?")
print("Answer:", result["answer"][:200])
print("Sources:", result["sources"])

Answer: Based on the context provided, Python is a high-level programming language created by Guido van Rossum in 1991.
Sources: ['python.txt', 'ml.txt']


## 7. Retrieval Evaluation

In [3]:
def evaluate_retrieval(queries, relevant_docs, retrieve_fn, docs, k=3):
    """Measure retrieval precision and recall."""
    precisions = []
    recalls = []
    
    for query, relevant in zip(queries, relevant_docs):
        retrieved = retrieve_fn(query, docs, k=k)
        retrieved_sources = set(d.metadata["source"] for d in retrieved)
        relevant_set = set(relevant)
        
        precision = len(retrieved_sources & relevant_set) / len(retrieved_sources) if retrieved_sources else 0
        recall = len(retrieved_sources & relevant_set) / len(relevant_set) if relevant_set else 0
        
        precisions.append(precision)
        recalls.append(recall)
    
    return {
        "precision": sum(precisions) / len(precisions),
        "recall": sum(recalls) / len(recalls)
    }

queries = ["What is Python?", "What is deep learning?", "Tell me about LangChain"]
relevant = [["python.txt"], ["dl.txt"], ["lc.txt"]]

metrics = evaluate_retrieval(queries, relevant, retrieve, docs, k=2)
print("Retrieval Precision:", round(metrics["precision"], 4))
print("Retrieval Recall:", round(metrics["recall"], 4))

Retrieval Precision: 0.3333
Retrieval Recall: 0.6667


## 8. Answer Faithfulness Evaluation

In [4]:
faithfulness_prompt = ChatPromptTemplate.from_template(
    "Is the answer faithful to the context? Answer YES or NO and explain.\n\nContext: {context}\nQuestion: {question}\nAnswer: {answer}"
)

faithfulness_chain = faithfulness_prompt | llm | StrOutputParser()

def evaluate_faithfulness(question, answer, context):
    result = faithfulness_chain.invoke({
        "context": context,
        "question": question,
        "answer": answer
    })
    return result

# Test
context = "Python is a programming language created by Guido van Rossum."
answer = "Python was created by Guido van Rossum in 1991."
result = evaluate_faithfulness("Who created Python?", answer, context)
print("Faithfulness:", result[:200])

Faithfulness: NO. The answer is not faithful to the context because it includes the year "1991," which is not mentioned in the provided context. The context only states that Python was created by Guido van Rossum, 


## 9. Common Evaluation Metrics

In [5]:
print("RAG Evaluation Metrics:")
print("")
print("Retrieval Metrics:")
print("  - Precision: % of retrieved docs that are relevant")
print("  - Recall: % of relevant docs that were retrieved")
print("  - MRR: Mean Reciprocal Rank of first relevant doc")
print("")
print("Generation Metrics:")
print("  - Faithfulness: Is the answer grounded in context?")
print("  - Relevance: Does the answer address the question?")
print("  - Correctness: Is the answer factually correct?")

RAG Evaluation Metrics:

Retrieval Metrics:
  - Precision: % of retrieved docs that are relevant
  - Recall: % of relevant docs that were retrieved
  - MRR: Mean Reciprocal Rank of first relevant doc

Generation Metrics:
  - Faithfulness: Is the answer grounded in context?
  - Relevance: Does the answer address the question?
  - Correctness: Is the answer factually correct?


## 10. Common Mistakes

1. Not evaluating retrieval separately from generation
2. Using only one metric
3. Not testing edge cases
4. Ignoring hallucination detection

## 11. Coding Exercises

### Exercise 1: Build Evaluation Suite
Create a complete evaluation pipeline.

### Exercise 2: Compare Retrievers
Compare different retrieval strategies.

In [6]:
# EXERCISE 1
print("Exercise: Build a RAG evaluation suite.")

Exercise: Build a RAG evaluation suite.


In [7]:
# EXERCISE 2
print("Exercise: Compare retrieval strategies.")

Exercise: Compare retrieval strategies.


## 12. Closed-Book Recall

1. What are the two parts of RAG evaluation?
2. What is faithfulness?
3. How do you measure retrieval quality?

## 13. Summary

RAG evaluation measures retrieval quality and answer faithfulness. Use multiple metrics. Test edge cases. Compare different strategies.

## Verification Status
```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: [langchain, langchain-groq]
OUTPUTS: PASS
LAST VERIFIED: 2026-08-30
```